In [1]:
import sys
from pathlib import Path

In [2]:
# in jupyter (lab / notebook), based on notebook path
module_path = str(Path.cwd().parents[0])

# # in standard python
# module_path = str(Path.cwd(__file__).parents[0] / "py")

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import wandb

import utils.helpers as helpers
import utils.push_to_hub as push_to_hub
import pytorch_lightning as pl

from transformers import (VisionEncoderDecoderConfig,
                          VisionEncoderDecoderModel,
                          DonutProcessor, 
                          DonutImageProcessor, 
                          XLMRobertaTokenizerFast)

from data.donut_dataset import data_loader, DonutDataset
from models.donut_pytorch_lightning import DonutModelPLModule
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
image_path = "/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_1280x960"
max_length = 768

### Load new data

In [5]:
dataset = data_loader(image_path)

print('Data Loading completes.')

this is the dataset DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 32165
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 6893
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 6893
    })
})
Data Loading completes.


### Load vision encoder decoder model

In [6]:
ori_size = helpers.image_size(dataset)
image_size = ori_size[::-1]

config = VisionEncoderDecoderConfig.from_pretrained('/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/models/donut-base')
config.encoder.image_size = image_size
config.decoder.max_length = max_length

processor = DonutProcessor.from_pretrained('/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/models/donut-base')
model = VisionEncoderDecoderModel(config=config)

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


In [7]:
model

VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0-1): 2 x DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
           

### Customize new dataset

In [ ]:
train_dataset = DonutDataset(image_path, max_length=max_length,
                             split="train", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model, 
                             processor=processor,
                             )

print("Donut Training Dataset is loaded.")

val_dataset = DonutDataset(image_path, max_length=max_length,
                             split="validation", task_start_token="<s_herbarium>", 
                             prompt_end_token="<s_herbarium>",
                             sort_json_key=False, # cord dataset is preprocessed -> no need for this
                             model=model,
                             processor=processor,
                             )

print("Donut Validation Dataset is loaded.")

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Donut Training Dataset is loaded.
Donut Validation Dataset is loaded.


In [ ]:
# the vocab size attribute stays constants (might be a bit unintuitive 
# - but doesn't include special tokens)
print("Original number of tokens:", processor.tokenizer.vocab_size)
print("Number of tokens after adding special tokens:", len(processor.tokenizer))

Original number of tokens: 57522
Number of tokens after adding special tokens: 57538


In [ ]:
processor.decode([57537])

'<s_herbarium>'

### Set up configurations of processor and model

In [ ]:
processor.image_processor.size = helpers.image_size(dataset)
processor.image_processor.do_align_long_axis = False

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_herbarium>'])[0]
model.config.hidden_dropout_prob = 0.2
model.config.attention_probs_dropout_prob = 0.2

# Accessing the model configuration
model_configuration = model.config

print("Model and processor settings are completed.")

Model and processor settings are completed.


In [ ]:
# sanity check
print("Pad token ID:", processor.decode([model.config.pad_token_id]))
print("Decoder start token ID:", processor.decode([model.config.decoder_start_token_id]))

Pad token ID: <pad>
Decoder start token ID: <s_herbarium>


### Set up PyTorch Lightning model

In [ ]:
PL_config = {
    'max_epochs': 6,
    'val_check_interval': 0.25,
    'check_val_every_n_epoch': 1,
    'gradient_clip_val': 1.0,
    'num_training_samples_per_epoch': 32165,
    'lr': 2.5e-5, # or 2e-5
    'weight_decay': 2e-5,
    'dropout_rate': 0.2,
    'train_batch_sizes': 10,
    'val_batch_sizes': 10,
    'num_nodes': 1,
    'warmup_steps': 2500,
    'result_path': "/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results",
    'verbose': True, 
    'seed': 16, 
    'num_workers' : 4
}

In [ ]:
model_lightning = DonutModelPLModule(PL_config, processor, model, 
                                     train_dataset, val_dataset)
print('PyTorch Lightning Model has been set up.')

PyTorch Lightning Model has been set up.


### Initialize a run of wandb

In [ ]:
wandb.init(mode="offline")
wandb_logger = WandbLogger(project="HeR-T-trial-241113", 
                           name="scratch_training")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


### Training!

In [ ]:
# use default patiente
early_stop_callback = EarlyStopping(monitor="val_edit_distance", 
                                    verbose=True, mode="min")
pushToHub = push_to_hub.PushToHubCallback()

In [ ]:
trainer = pl.Trainer(
        accelerator="gpu",
        devices=4,
        # strategy="xla_debug",
        max_epochs=PL_config['max_epochs'],
        val_check_interval=PL_config['val_check_interval'],
        check_val_every_n_epoch=PL_config['check_val_every_n_epoch'],
        gradient_clip_val=PL_config['gradient_clip_val'],
        precision='bf16-mixed', # we'll use mixed precision
        num_sanity_val_steps=0,
        logger=wandb_logger,
        callbacks=[pushToHub, early_stop_callback]
)

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/p ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model_lightning)

You are using a CUDA device ('NVIDIA A100-SXM-64GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
You are using a CUDA device ('NVIDIA A100-SXM-64GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
You are using a CUDA device ('NVIDIA A100-SXM-64GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/ge

Epoch 0:   0%|                                                   | 0/805 [00:00<?, ?it/s]

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/PIL/TiffImagePlugin.py:870: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 0:  25%|███████▏                     | 201/805 [10:58<32:59,  0.31it/s, v_num=5wh1]
Validation: |                                                      | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:   0%|                                   | 0/173 [00:00<?, ?it/s]

/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/pytorch_lightning/utilities/data.py:77: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 10. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


...


Epoch 0:  50%|██████████████▍              | 402/805 [57:15<57:24,  0.12it/s, v_num=5wh1]
Validation: |                                                      | 0/? [00:00<?, ?it/s]
Epoch 0:  75%|████████████████████▏      | 603/805 [2:05:34<42:03,  0.08it/s, v_num=5wh1]
Validation: |                                                      | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████████████████████▉| 804/805 [2:27:46<00:11,  0.09it/s, v_num=5wh1]
Validation: |                                                      | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████████████████████▉| 804/805 [2:41:00<00:12,  0.08it/s, v_num=5wh1]Saving model to the local, epoch 0Saving model to the local, epoch 0Saving model to the local, epoch 0


Epoch 1:  25%|███████▏                     | 201/805 [10:19<31:02,  0.32it/s, v_num=5wh1]
Validation: |                                                      | 0/? [00:00<?, ?it/s]
Epoch 1:  50%|██████████████▍              | 402/805 [37:28<37:33,  0.18it/s, v_num=5

`Trainer.fit` stopped: `max_epochs=6` reached.


Saving model to the local after training
Epoch 5: 100%|███████████████████████████| 805/805 [2:10:58<00:00,  0.10it/s, v_num=5wh1]


[rank: 0] Received SIGTERM: 15


ProcessExitedException: process 3 terminated with signal SIGKILL

### Summarize the run of wandb

In [19]:
wandb.finish()

epoch,▁▁▁▁▁▁▁▂▂▂▂▂▂▂▄▄▄▄▄▄▅▅▅▅▅▅▅▇▇▇▇▇▇███████
train_loss,█▆▄▄▄▄▃▃▃▃▂▃▃▂▃▂▂▂▂▂▁▂▂▂▂▂▂▂▁▁▂▂▁▂▂▁▁▂▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
val_accuracy,▁▃████▇██████████▆███▇▆▇
val_edit_distance,█▆▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▃▁▁▁▂▃▂
epoch,5
train_loss,1.82054
trainer/global_step,4828
val_accuracy,0.62929
val_edit_distance,0.37071
